# Chapter 2 — makemore: a language model with 27 letters

From *Neural Networks: Zero to Hero — The Textbook*.

Run each cell with **Shift+Enter**. Before you run one, say out loud what you expect it to print; being wrong is the useful part.


## Chapter 2 — makemore: a language model with 27 letters

**Video:** 1h57m · [youtu.be/PaCmpygFfXo](https://youtu.be/PaCmpygFfXo) · **New:** PyTorch tensors, probability distributions, and a demonstration that counting and learning can reach the same answer.

### The problem

Invent new human names. Train on 32,033 real ones and generate plausible fakes. The project name says it: **makemore** makes more of whatever you feed it.

> **Say it to a six-year-old.** You read a big list of names and notice that after the letter Q there is almost always a U, and that lots of names end in A. Once you know which letters like to sit next to which other letters, you can make up brand new names that sound real. That is the whole trick. The computer just counts much faster than you.

### Step 1 — load and look at the data

Never train on data you have not looked at.

**Run it.**

In [ ]:
words = open('names.txt', 'r').read().splitlines()
print("number of names:", len(words))
print("first five:", words[:5])
print("shortest:", min(len(w) for w in words), "| longest:", max(len(w) for w in words))

**What you should see:**

**Expected output:**

```
number of names: 32033
first five: ['emma', 'olivia', 'ava', 'isabella', 'sophia']
shortest: 2 | longest: 15
```

[verified]

### Step 2 — frame it as next-character prediction

The word `emma` becomes five training examples, using a special token `.` to mark both start and end: `.→e`, `e→m`, `m→m`, `m→a`, `a→.` [transcript]. A pair of adjacent characters is a **bigram**.

**Why one token for both start and end.** You need to model two things: which letters begin a name, and when a name is finished. Using the same symbol for both keeps the table square at 27×27 and costs nothing, because the start position and the end position never get confused: `.` as an input means "we are at the beginning," and `.` as an output means "stop."

**Run it.** Build the character-to-integer mapping:

In [ ]:
chars = sorted(list(set(''.join(words))))
stoi = {s: i+1 for i, s in enumerate(chars)}   # a=1, b=2, ... z=26
stoi['.'] = 0                                  # the special token gets 0
itos = {i: s for s, i in stoi.items()}         # the reverse mapping
print("vocabulary size:", len(itos))
print("a ->", stoi['a'], "| z ->", stoi['z'], "| . ->", stoi['.'])

**What you should see:**

**Expected output:**

```
vocabulary size: 27
a -> 1 | z -> 26 | . -> 0
```

[verified]

### Step 3 — count every pair

**Run it.**

In [ ]:
import torch
N = torch.zeros((27, 27), dtype=torch.int32)
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        N[stoi[ch1], stoi[ch2]] += 1

print("total bigrams counted:", N.sum().item())
print("names starting with m:", N[0, stoi['m']].item())
print("times 'q' is followed by 'u':", N[stoi['q'], stoi['u']].item())
print("times 'j' is followed by 'q':", N[stoi['j'], stoi['q']].item())

**What you should see:**

**Expected output:**

```
total bigrams counted: 228146
names starting with m: 2538
times 'q' is followed by 'u': 206
times 'j' is followed by 'q': 0
```

[verified]

`zip(chs, chs[1:])` is the standard Python idiom for "every adjacent pair": it pairs the list with a copy of itself shifted by one. 2,538 names out of 32,033 start with `m`, just under 8%, which is the "about 2,500, a bit less than 10%" from the lecture [transcript].

Hold on to that `jq` count of zero. It causes an infinity in a moment.

### Step 4 — turn counts into probabilities

**Run it.**

In [ ]:
P = (N + 1).float()               # the +1 is smoothing; see step 6
P /= P.sum(1, keepdim=True)       # keepdim=True, per section 1.9
print("row for 'q' sums to:", P[stoi['q']].sum().item())
print("P(u | q) =", round(P[stoi['q'], stoi['u']].item(), 4))
print("P(a | q) =", round(P[stoi['q'], stoi['a']].item(), 4))

**What you should see:**

**Expected output:**

```
row for 'q' sums to: 1.0
P(u | q) = 0.6923
P(a | q) = 0.0468
```

[verified]

Read `P(u | q)` as "the probability of u given q." After a `q`, this model says `u` 69% of the time and `a` under 5%, which is a real fact about English names that nobody programmed in. It fell out of counting 206 occurrences of `qu`.

### Step 5 — sample new names

**Run it.**

In [ ]:
g = torch.Generator().manual_seed(2147483647)   # fixed seed, so you get my exact output
for _ in range(5):
    out, ix = [], 0
    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        if ix == 0:
            break
        out.append(itos[ix])
    print(''.join(out))

**What you should see:**

**Expected output:**

```
cexze
momasurailezitynn
konimittain
llayn
ka
```

[verified]

`torch.multinomial` draws from a probability distribution: hand it `[0.6, 0.3, 0.1]` and it returns index 0 about 60% of the time. The **generator with a fixed seed** makes randomness reproducible, so you and I get identical names. Change the seed and you get different ones; remove it and you get different ones every run.

These are bad names, in an instructive way. `ka` and `llayn` are name-shaped. `momasurailezitynn` is not. The model only ever knows the single preceding letter, so it cannot remember that it started a name eleven letters ago.

### Step 6 — score the model, and meet infinity

**Run it.**

In [ ]:
log_likelihood, n = 0.0, 0
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        log_likelihood += torch.log(P[stoi[ch1], stoi[ch2]])
        n += 1
print(f"average negative log likelihood (the loss): {-log_likelihood/n:.4f}")

**What you should see:**

**Expected output:**

```
average negative log likelihood (the loss): 2.4544
```

[verified] [transcript]

Compare against 3.2958, the score of a model that knows nothing (section 1.12). The counting model learned something real.

**Now the infinity.** Build the same table without the `+1`:

In [ ]:
P0 = N.float()
P0 /= P0.sum(1, keepdim=True)
ll, m = 0.0, 0
for ch1, ch2 in zip("andrejq", "ndrejq"):
    ll += torch.log(P0[stoi[ch1], stoi[ch2]]); m += 1
print("loss on the name 'andrejq':", (-ll/m).item())

**What you should see:**

**Expected output:**

```
loss on the name 'andrejq': inf
```

[verified] [transcript]

`jq` never occurred, so the model assigns it probability exactly 0, `log(0)` is `−inf`, and the loss on any name containing `jq` is infinite. The model is claiming that this name is not merely unlikely but *impossible*, which is an absurd thing to be certain about.

**The fix is the `+1`,** called **smoothing**: add a fake count of 1 to every cell before normalizing, so nothing is ever impossible. The cost is a slightly worse average loss; the benefit is that a single unseen pair no longer destroys everything.

**Analogy.** Insurance. You pay a small premium on every prediction to avoid one unbounded loss.

> **For the PhD in the room.** Add-one smoothing is Laplace smoothing, equivalent to a symmetric Dirichlet(α=1) prior over the categorical parameters with the posterior mean as the estimate. In the language-modeling literature it is known to be a poor smoother compared to Kneser-Ney, which redistributes mass according to continuation counts rather than uniformly. Nobody uses it seriously at this point; the neural approach in step 7 is precisely the escape from smoothing schemes, since parameter sharing through embeddings performs the smoothing implicitly and in a data-dependent way.

### Step 7 — the same model, learned instead of counted

Now throw the counts away and train a network to do the same job.

**One-hot encoding.** A network takes numbers, not letters. Represent letter 13 as a 27-long vector of zeros with a single 1 in position 13. [standard]

**Run it.**

In [ ]:
import torch.nn.functional as F
xs, ys = [], []
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        xs.append(stoi[ch1]); ys.append(stoi[ch2])
xs, ys = torch.tensor(xs), torch.tensor(ys)
num = xs.nelement()
print("number of training examples:", num)

xenc = F.one_hot(xs[:3], num_classes=27).float()
print("first three inputs as one-hot rows, shape:", tuple(xenc.shape))
print(xenc[0][:8], "...")

**What you should see:**

**Expected output:**

```
number of training examples: 228146
first three inputs as one-hot rows, shape: (3, 27)
tensor([1., 0., 0., 0., 0., 0., 0., 0.]) ...
```

[verified]

228,146 examples, the same number as the bigram count, because every bigram is one training example.

**The key insight about one-hot.** Multiplying a one-hot row by a matrix **selects a row of that matrix**. Nothing else. So `xenc @ W` where `W` is 27×27 is an elaborate way of looking up row 13, which means this "neural network" is a lookup table wearing a costume. Understanding that makes the punchline land.

**Run it.** The training loop:

In [ ]:
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

for k in range(200):
    xenc = F.one_hot(xs, num_classes=27).float()   # (228146, 27)
    logits = xenc @ W                              # (228146, 27) raw scores
    counts = logits.exp()                          # softmax, step 1
    probs = counts / counts.sum(1, keepdim=True)   # softmax, step 2
    loss = -probs[torch.arange(num), ys].log().mean() + 0.01*(W**2).mean()
    if k % 50 == 0 or k == 199:
        print(f"step {k:3d} loss {loss.item():.4f}")
    W.grad = None          # reset gradients (the += rule from Chapter 1)
    loss.backward()        # PyTorch does what your engine did
    W.data += -50 * W.grad # step downhill

**What you should see:**

**Expected output:**

```
step   0 loss 3.7686
step  50 loss 2.5099
step 100 loss 2.4900
step 150 loss 2.4849
step 199 loss 2.4830
```

[verified]

Line by line, because every piece of this recurs for the rest of the book:

- `probs[torch.arange(num), ys]` plucks, for each of the 228,146 rows, the probability assigned to the character that actually came next. `torch.arange(num)` is `[0,1,2,...]`, so this pairs row 0 with `ys[0]`, row 1 with `ys[1]`, and so on. This double-indexing trick appears constantly.
- `.log().mean()` and the minus sign make it the negative log likelihood of section 1.12.
- `+ 0.01*(W**2).mean()` is **regularization**: a penalty for large weights, which pushes them toward zero. It is the gradient-descent equivalent of adding 1 to every count. Two different-looking hacks, one idea: prevent extreme confidence. [transcript]
- `-50` is a very large learning rate, tolerable here because the model is tiny and the loss surface is simple.

### Step 8 — the punchline

The network converges to about **2.48** after 200 steps and continues toward **2.45** with more; the counting table gave **2.4544**. Same answer, reached two completely different ways. [verified]

Karpathy is explicit about why: "fundamentally, we're not taking any additional information" [transcript]. The model can express exactly one thing, the probability of each letter given the previous letter, so both methods find the same optimum. Counting finds it in one pass; gradient descent walks to it.

**Then why bother with the hard way?** Because counting does not scale. One character of context needs a 27×27 table of 729 cells. Ten characters of context needs 27¹⁰ cells.

**Run it.**

In [ ]:
print(f"{27**10:,} cells needed for a 10-character-context table")
print(f"{27**10 * 4 / 1e12:,.0f} terabytes just to store it as 32-bit floats")

**What you should see:**

**Expected output:**

```
205,891,132,094,649 cells needed for a 10-character-context table
824 terabytes just to store it as 32-bit floats
```

[verified]

And nearly all of those cells would be zero, because you have only 228,146 examples to fill 205 trillion cells. Karpathy: "we can't actually keep everything in a table anymore, so this is fundamentally an unscalable approach" [transcript]. The neural network is the escape route, and Chapter 3 takes it.

### Exercises

1. **Trigrams.** Build a model that uses the *two* preceding characters. The table becomes 27×27×27, which is 19,683 cells and still feasible. Does the loss beat 2.4544? (It should reach roughly 2.2.)
2. **Try the wrong normalization.** Replace `P.sum(1, keepdim=True)` with `P.sum(0, keepdim=True)` and see how badly the samples degrade. This is the section 1.9 bug, in a real setting.
3. **Vary the smoothing.** Try `+0.001` and `+100` instead of `+1`. Watch the loss get better and the samples get worse, then the reverse. Explain why.
4. **Delete the regularization term** in the neural version and train for 500 steps. The loss will go slightly lower. Explain, using the smoothing discussion, why that is not automatically an improvement.

### Troubleshooting

| Symptom | Cause |
|---|---|
| Loss is `nan` | A probability hit exactly 0 and `log(0)` is `-inf`; add smoothing, or use `F.cross_entropy` which is numerically safer |
| Samples are all one letter | Rows normalized along the wrong axis (section 1.9) |
| `IndexError` in `stoi[ch]` | Your `names.txt` contains a stray character such as a capital or a hyphen; check `sorted(set(''.join(words)))` |
| Loss stuck at 3.29 | Learning rate far too small, or you forgot `loss.backward()` |
| Different samples than this book | The generator seed differs, or you called the generator a different number of times before sampling |

### 30-second version

Count how often each letter follows each other letter and you have a working language model that scores 2.4544. Then throw the counts away, train a one-layer neural network on the same task, and it lands on the same number, because it is the same model learned a harder way. The counting version cannot grow past one letter of memory: ten letters of context would need a table of 205 trillion cells. The network version can grow, and the rest of the course is that growth.

---